In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [2]:

# BRONZE - Incremental ingestion with source change detection

import os
import json
import requests
from datetime import datetime
from notebookutils import mssparkutils

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
BRONZE_DIR = "Files/bronze/yellow_taxi"
STAGING_DIR = "Files/bronze/yellow_taxi_staging"
MANIFEST_PATH = "Files/metadata/yellow_taxi_source_manifest.json"
DOWNLOAD_DIR = "/tmp/nyc_taxi_bronze"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
mssparkutils.fs.mkdirs(BRONZE_DIR)
mssparkutils.fs.mkdirs(STAGING_DIR)
mssparkutils.fs.mkdirs("Files/metadata")

# Load the existing manifest
if mssparkutils.fs.exists(MANIFEST_PATH):
    manifest = json.loads(
        mssparkutils.fs.head(MANIFEST_PATH, 10 * 1024 * 1024)
    )
else:
    manifest = {}

now = datetime.now()
available_files = []
source_metadata = {}
failed = []

# Discover available monthly files and their metadata
for year in range(2025, now.year + 1):
    last_month = now.month if year == now.year else 12

    for month in range(1, last_month + 1):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{BASE_URL}/{filename}"

        try:
            response = requests.head(
                url, timeout=30, allow_redirects=True
            )

            if response.status_code in (403, 404):
                print(f"Unavailable or unverified: {filename}")
                continue

            response.raise_for_status()

            metadata = {
                "etag": response.headers.get("ETag"),
                "last_modified": response.headers.get("Last-Modified"),
                "content_length": response.headers.get("Content-Length")
            }

            if not any(metadata.values()):
                raise RuntimeError(
                    f"No usable source metadata returned for {filename}"
                )

            available_files.append(filename)
            source_metadata[filename] = metadata

        except Exception as e:
            failed.append(f"HEAD {filename}: {str(e)}")

# Do not change Bronze if source discovery failed
if failed:
    raise RuntimeError(
        "Source discovery failed. No Bronze changes were made:\n"
        + "\n".join(failed)
    )

if not available_files:
    raise RuntimeError("No source files were discovered.")

downloaded = 0
skipped = 0
initialized = 0
updated_manifest = dict(manifest)

# Process every discovered file
for filename in sorted(available_files):
    target = f"{BRONZE_DIR}/{filename}"
    staging = f"{STAGING_DIR}/{filename}.download"
    backup = f"{STAGING_DIR}/{filename}.backup"
    local_path = os.path.join(DOWNLOAD_DIR, filename)
    url = f"{BASE_URL}/{filename}"

    current = source_metadata[filename]
    previous = manifest.get(filename)
    target_exists = mssparkutils.fs.exists(target)

    if target_exists and previous is None:
        print(
            f"Existing Bronze file has no manifest entry: {filename}. "
            "It will be downloaded and verified."
        )

    # Detect metadata changes, including previously present fields
    # that are now missing.
    changed = False

    if previous is not None:
        for key in previous:
            if (
                previous.get(key) is not None
                and current.get(key) is None
            ):
                changed = True
                break

            if (
                current.get(key) is not None
                and current.get(key) != previous.get(key)
            ):
                changed = True
                break

    # Skip only if the file exists and its metadata is unchanged
    if target_exists and previous is not None and not changed:
        skipped += 1
        continue

    try:
        # Download to local temporary storage
        with requests.get(
            url, stream=True, timeout=300
        ) as response:
            response.raise_for_status()

            with open(local_path, "wb") as f:
                for chunk in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)

        actual_size = os.path.getsize(local_path)
        expected_size = current.get("content_length")

        if expected_size and actual_size != int(expected_size):
            raise RuntimeError(
                f"Size mismatch: expected {expected_size}, "
                f"downloaded {actual_size}"
            )

        if actual_size == 0:
            raise RuntimeError("Downloaded file is empty")

        # Upload to staging before replacing the Bronze copy
        if mssparkutils.fs.exists(staging):
            mssparkutils.fs.rm(staging, False)

        mssparkutils.fs.cp(f"file:{local_path}", staging)

        if not mssparkutils.fs.exists(staging):
            raise RuntimeError("Staging upload failed")

        had_target = mssparkutils.fs.exists(target)

        # Remove an old backup only after staging is ready
        if mssparkutils.fs.exists(backup):
            mssparkutils.fs.rm(backup, False)

        if had_target:
            mssparkutils.fs.mv(target, backup, True, True)

        try:
            mssparkutils.fs.mv(staging, target, True, True)

            if not mssparkutils.fs.exists(target):
                raise RuntimeError("Bronze replacement failed")

            if had_target and mssparkutils.fs.exists(backup):
                mssparkutils.fs.rm(backup, False)

        except Exception:
            # Restore the previous file if replacement failed
            if (
                not mssparkutils.fs.exists(target)
                and mssparkutils.fs.exists(backup)
            ):
                mssparkutils.fs.mv(backup, target, True, True)
            raise

        updated_manifest[filename] = current
        downloaded += 1

        if previous is None:
            initialized += 1
            action = "Added"
        else:
            action = "Updated"

        print(f"{action}: {filename} ({actual_size:,} bytes)")

    except Exception as e:
        failed.append(f"{filename}: {str(e)}")

    finally:
        if os.path.exists(local_path):
            os.remove(local_path)

print("\nBronze ingestion summary")
print("Available source files:", len(available_files))
print("Downloaded or updated:", downloaded)
print("Skipped unchanged files:", skipped)
print("Manifest entries initialized:", initialized)
print("Failed:", len(failed))

# Publish metadata only if every file operation succeeded
if failed:
    raise RuntimeError(
        "Bronze ingestion failed. Manifest was not updated:\n"
        + "\n".join(failed)
    )

mssparkutils.fs.put(
    MANIFEST_PATH,
    json.dumps(updated_manifest, indent=2),
    True
)

print("Bronze ingestion completed successfully")


StatementMeta(, ce0df470-bf64-4663-b78f-6df94055ba20, 4, Finished, Available, Finished, False)

Unavailable or unverified: yellow_tripdata_2026-09.parquet
Unavailable or unverified: yellow_tripdata_2026-10.parquet

Bronze ingestion completed
Available source files: 20
Downloaded or updated: 0
Skipped unchanged files: 20
Manifest entries initialized: 20
Failed: 0


In [5]:

# SILVER - Incremental processing with file-level replacement

import json
from pyspark.sql import functions as F
from delta.tables import DeltaTable

SILVER_TABLE = "yellow_taxi_silver"
BRONZE_MANIFEST_PATH = "Files/metadata/yellow_taxi_source_manifest.json"
SILVER_MANIFEST_PATH = "Files/metadata/yellow_taxi_silver_manifest.json"

# Read the Bronze source manifest
if not mssparkutils.fs.exists(BRONZE_MANIFEST_PATH):
    raise RuntimeError("Bronze source manifest not found.")

bronze_manifest = json.loads(
    mssparkutils.fs.head(BRONZE_MANIFEST_PATH, 10 * 1024 * 1024)
)

bronze_files = sorted([
    item.name
    for item in mssparkutils.fs.ls(BRONZE_DIR)
    if item.name.endswith(".parquet")
])

if not bronze_files:
    raise RuntimeError("Bronze contains no Parquet files.")

# Load Silver processing history
silver_manifest_exists = mssparkutils.fs.exists(
    SILVER_MANIFEST_PATH
)

if silver_manifest_exists:
    silver_manifest = json.loads(
        mssparkutils.fs.head(
            SILVER_MANIFEST_PATH, 10 * 1024 * 1024
        )
    )
else:
    silver_manifest = {"files": {}, "total_rows": 0}

silver_history = silver_manifest.get("files", {})
table_exists = spark.catalog.tableExists(SILVER_TABLE)

# First run after migration: rebuild Silver with file lineage.
full_rebuild = not silver_manifest_exists or not table_exists

if full_rebuild:
    files_to_process = bronze_files
else:
    files_to_process = [
        filename for filename in bronze_files
        if (
            filename not in silver_history
            or silver_history[filename].get("source_metadata")
            != bronze_manifest.get(filename)
        )
    ]

# Detect files removed from Bronze.
removed_files = sorted(set(silver_history) - set(bronze_files))

if removed_files:
    raise RuntimeError(
        "Bronze files were removed. Reconcile Silver before proceeding: "
        + ", ".join(removed_files)
    )

if not files_to_process:
    print("Silver is already up to date. No files require processing.")
else:
    processed_frames = []
    updated_file_stats = {}

    for filename in files_to_process:
        path = f"{BRONZE_DIR}/{filename}"

        file_df = (
            spark.read.parquet(path)
            .withColumn("source_file", F.lit(filename))
        )

        valid_df = file_df.filter(
            F.col("tpep_dropoff_datetime")
            >= F.col("tpep_pickup_datetime")
        )

        source_count = file_df.count()
        valid_count = valid_df.count()

        processed_frames.append(valid_df)

        updated_file_stats[filename] = {
            "source_metadata": bronze_manifest.get(filename),
            "source_rows": source_count,
            "silver_rows": valid_count
        }

    changed_df = processed_frames[0]

    for frame in processed_frames[1:]:
        changed_df = changed_df.unionByName(
    frame,
    allowMissingColumns=True
)

    if full_rebuild:
        (
            changed_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("source_file")
            .saveAsTable(SILVER_TABLE)
        )
    else:
        # Replace only the source files that were reprocessed.
        quoted_files = [
            "'" + name.replace("'", "''") + "'"
            for name in files_to_process
        ]

        predicate = (
            "source_file IN (" + ", ".join(quoted_files) + ")"
        )

        (
            changed_df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", predicate)
            .saveAsTable(SILVER_TABLE)
        )

    # Validate the affected files after persistence.
    persisted_affected = (
        spark.table(SILVER_TABLE)
        .filter(F.col("source_file").isin(files_to_process))
        .groupBy("source_file")
        .count()
        .collect()
    )

    persisted_counts = {
        row["source_file"]: row["count"]
        for row in persisted_affected
    }

    for filename in files_to_process:
        expected = updated_file_stats[filename]["silver_rows"]
        actual = persisted_counts.get(filename, 0)

        if actual != expected:
            raise RuntimeError(
                f"Silver validation failed for {filename}: "
                f"expected {expected}, found {actual}"
            )

    # Update processing history only after successful validation.
    new_history = dict(silver_history)
    new_history.update(updated_file_stats)

    total_rows = sum(
        item["silver_rows"] for item in new_history.values()
    )

    new_manifest = {
        "files": new_history,
        "total_rows": total_rows,
        "last_updated_utc": datetime.utcnow().isoformat() + "Z"
    }

    mssparkutils.fs.put(
        SILVER_MANIFEST_PATH,
        json.dumps(new_manifest, indent=2),
        True
    )

    print("Silver processing completed")
    print("Full rebuild:", full_rebuild)
    print("Files processed:", len(files_to_process))
    print("Files unchanged:", len(bronze_files) - len(files_to_process))
    print("Rows in processed files:", changed_df.count())
    print("Silver rows recorded in manifest:", total_rows)



StatementMeta(, ce0df470-bf64-4663-b78f-6df94055ba20, 7, Finished, Available, Finished, False)

Silver is already up to date. No files require processing.


In [13]:

# GOLD - Incremental analytical table refresh

import json
from datetime import datetime, timedelta
from pyspark.sql import functions as F
spark.conf.set("spark.sql.cbo.enabled", "false")
GOLD_TABLE = "yellow_taxi_gold"
SILVER_TABLE = "yellow_taxi_silver"
SILVER_MANIFEST_PATH = "Files/metadata/yellow_taxi_silver_manifest.json"
GOLD_MANIFEST_PATH = "Files/metadata/yellow_taxi_gold_manifest.json"

GOLD_START = None
GOLD_RULE_VERSION = 2

# Dynamic upper boundary: first day of next month
month_start = datetime.now().replace(
    day=1, hour=0, minute=0, second=0, microsecond=0
)
next_month = (month_start + timedelta(days=32)).replace(day=1)
GOLD_END_EXCLUSIVE = next_month.strftime("%Y-%m-%d")

# Read Silver processing history
if not mssparkutils.fs.exists(SILVER_MANIFEST_PATH):
    raise RuntimeError("Silver manifest not found.")

silver_manifest = json.loads(
    mssparkutils.fs.head(
        SILVER_MANIFEST_PATH, 10 * 1024 * 1024
    )
)

silver_history = silver_manifest.get("files", {})

if not silver_history:
    raise RuntimeError("Silver manifest contains no file history.")

# Load Gold processing history
gold_manifest_exists = mssparkutils.fs.exists(
    GOLD_MANIFEST_PATH
)

if gold_manifest_exists:
    gold_manifest = json.loads(
        mssparkutils.fs.head(
            GOLD_MANIFEST_PATH, 10 * 1024 * 1024
        )
    )
else:
    gold_manifest = {"files": {}}

gold_history = gold_manifest.get("files", {})
table_exists = spark.catalog.tableExists(GOLD_TABLE)

# Initial migration or recovery: rebuild Gold completely
full_rebuild = (
    not gold_manifest_exists
    or not table_exists
    or gold_manifest.get("rule_version") != GOLD_RULE_VERSION
    or gold_manifest.get("gold_start") != GOLD_START
)

if full_rebuild:
    files_to_process = sorted(silver_history.keys())
else:
    files_to_process = [
        filename
        for filename, silver_info in silver_history.items()
        if (
            filename not in gold_history
            or gold_history[filename].get("source_metadata")
            != silver_info.get("source_metadata")
        )
    ]

silver_df = spark.table(SILVER_TABLE)

if full_rebuild:
    source_df = silver_df
elif files_to_process:
    source_df = silver_df.filter(
        F.col("source_file").isin(files_to_process)
    )
else:
    source_df = None

if not files_to_process:
    print("Gold is already up to date. No files require processing.")

else:
    gold_df = source_df.filter(
    F.col("tpep_pickup_datetime") < F.to_timestamp(F.lit(GOLD_END_EXCLUSIVE))
)

    expected_counts_rows = (
        gold_df.groupBy("source_file")
        .count()
        .collect()
    )

    expected_counts = {
        row["source_file"]: row["count"]
        for row in expected_counts_rows
    }

    if full_rebuild:
        (
            gold_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("source_file")
            .saveAsTable(GOLD_TABLE)
        )
    else:
        quoted_files = [
            "'" + name.replace("'", "''") + "'"
            for name in files_to_process
        ]

        predicate = (
            "source_file IN (" + ", ".join(quoted_files) + ")"
        )

        (
            gold_df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", predicate)
            .saveAsTable(GOLD_TABLE)
        )

    # Validate persisted counts for every affected source file
    persisted_rows = (
        spark.table(GOLD_TABLE)
        .filter(F.col("source_file").isin(files_to_process))
        .groupBy("source_file")
        .count()
        .collect()
    )

    persisted_counts = {
        row["source_file"]: row["count"]
        for row in persisted_rows
    }

    for filename in files_to_process:
        expected = expected_counts.get(filename, 0)
        actual = persisted_counts.get(filename, 0)

        if actual != expected:
            raise RuntimeError(
                f"Gold validation failed for {filename}: "
                f"expected {expected}, found {actual}"
            )

    # Update Gold history only after successful validation
    new_history = dict(gold_history)

    for filename in files_to_process:
        new_history[filename] = {
            "source_metadata": silver_history[filename].get(
                "source_metadata"
            ),
            "gold_rows": expected_counts.get(filename, 0)
        }

    new_manifest = {
        "files": new_history,
        "rule_version": GOLD_RULE_VERSION,
"gold_start": GOLD_START,
"gold_end_exclusive": GOLD_END_EXCLUSIVE,
        "last_updated_utc": datetime.utcnow().isoformat() + "Z"
    }

    mssparkutils.fs.put(
        GOLD_MANIFEST_PATH,
        json.dumps(new_manifest, indent=2),
        True
    )

    print("Gold processing completed")
    print("Full rebuild:", full_rebuild)
    print("Files processed:", len(files_to_process))
    print("Files unchanged:", len(silver_history) - len(files_to_process))
    print("Gold rows written for processed files:", gold_df.count())
    print("Gold date range:", GOLD_START, "to", GOLD_END_EXCLUSIVE)


StatementMeta(, ce0df470-bf64-4663-b78f-6df94055ba20, 15, Finished, Available, Finished, False)

Gold is already up to date. No files require processing.


In [1]:
print("Silver rows:", spark.table("yellow_taxi_silver").count())
print("Gold rows:", spark.table("yellow_taxi_gold").count())

StatementMeta(, 942d2e1d-f686-4d05-94a2-ab9d359cb1de, 3, Finished, Available, Finished, False)

Silver rows: 78423712
Gold rows: 78423712
